## 4.8: Grouping Data & Aggregating Variables

### Importing Libraries and Dataframes

In [3]:
# Import libraries 
import pandas as pd
import numpy as np
import os

In [5]:
# Set Path
path = '/Users/muhammaddildar/Desktop/03-2025 Instacart Basket Analysis'

In [7]:
ords_prods_merge = pd.read_pickle(os.path.join(path, 'Data', 'Prepared Data', 'ords_prods_merge_updated.pkl'))

In [9]:
ords_prods_merge.shape

(32432460, 19)

In [11]:
ords_prods_merge.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge,product_name,aisle_id,department_id,prices,price_range_loc,busiest_day,busiest_day_updated,slowest_day_updated,busiest_period_of_day
0,2539329,1,1,2,8,NaN,196,1,0,both,Soda,77,7,9.0,Mid-range product,Regularly busy,Busiest days,Regularly busy,Average orders
1,2539329,1,1,2,8,NaN,14084,2,0,both,Organic Unsweetened Vanilla Almond Milk,91,16,12.5,Mid-range product,Regularly busy,Busiest days,Regularly busy,Average orders
2,2539329,1,1,2,8,NaN,12427,3,0,both,Original Beef Jerky,23,19,4.4,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders
3,2539329,1,1,2,8,NaN,26088,4,0,both,Aged White Cheddar Popcorn,23,19,4.7,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders
4,2539329,1,1,2,8,NaN,26405,5,0,both,XL Pick-A-Size Paper Towel Rolls,54,17,1.0,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders


## Aggregate the Mean of order_number Grouped by department_id for the Entire Dataframe

In [15]:
# Group by 'department_id' and calculate the mean of 'order_number'
department_order_mean = ords_prods_merge.groupby('department_id').agg({'order_number': ['mean']})

# Display the result
department_order_mean


,order_number
,mean
department_id,
1,15.457687
2,17.277920
3,17.179756
4,17.811403
5,15.215751
6,16.439806
7,17.225773
8,15.340520


## Analysis of the Results:
The mean order_number for each department in the full dataframe is generally consistent with the subset, but slight differences may occur due to the smaller sample size in the subset.
The full dataframe provides a more accurate picture of the department-level order numbers, while the subset may show minor variations.

## Create a Loyalty Flag for Existing Customers

In [24]:
# Step 1: Set the loyalty flags
ords_prods_merge.loc[ords_prods_merge['max_order'] > 40, 'loyalty_flag'] = 'Loyal customer'
ords_prods_merge.loc[(ords_prods_merge['max_order'] <= 40) & (ords_prods_merge['max_order'] > 10), 'loyalty_flag'] = 'Regular customer'
ords_prods_merge.loc[ords_prods_merge['max_order'] <= 10, 'loyalty_flag'] = 'New customer'

# Step 2: Display the frequency of the 'loyalty_flag' column
ords_prods_merge['loyalty_flag'].value_counts(dropna=False)


loyalty_flag
Regular customer    15890123
Loyal customer      10293366
New customer         6248971
Name: count, dtype: int64

In [30]:
# Group by 'loyalty_flag' and calculate basic statistics for the 'prices' column
price_stats_by_loyalty = ords_prods_merge.groupby('loyalty_flag')['prices'].agg(
    ['mean', 'min', 'max', 'std', 'median']
)

# Display the statistics
price_stats_by_loyalty


,mean,min,max,std,median
loyalty_flag,,,,,
Loyal customer,10.388902,1.0,99999.0,327.870016,7.4
New customer,13.294907,1.0,99999.0,597.322095,7.4
Regular customer,12.496618,1.0,99999.0,539.494200,7.4


### To address this, we can filter out or remove the extreme outliers (such as the product priced at 99999) and then recalculate the average prices. This will provide a more accurate representation of spending patterns among the customers.

In [32]:
# Remove outliers where the price is 99999
ords_prods_merge_cleaned = ords_prods_merge[ords_prods_merge['prices'] != 99999]

# Recalculate the statistics on the cleaned data
price_stats_cleaned = ords_prods_merge_cleaned.groupby('loyalty_flag')['prices'].agg(
    ['mean', 'min', 'max', 'std', 'median']
)

# Display the updated statistics
price_stats_cleaned


,mean,min,max,std,median
loyalty_flag,,,,,
Loyal customer,9.582643,1.0,14900.0,163.957960,7.4
New customer,10.062732,1.0,14900.0,183.367021,7.4
Regular customer,9.897803,1.0,14900.0,176.658723,7.4


###  Create a Spending Flag Based on Average Price

In [38]:
# Step 1: Calculate the average price for each user
ords_prods_merge['avg_price'] = ords_prods_merge.groupby('user_id')['prices'].transform('mean')

# Step 2: Create the spending flag based on the average price
ords_prods_merge.loc[ords_prods_merge['avg_price'] < 10, 'spending_flag'] = 'Low spender'
ords_prods_merge.loc[ords_prods_merge['avg_price'] >= 10, 'spending_flag'] = 'High spender'

# Display the result to check the new 'spending_flag' column
ords_prods_merge['spending_flag'].value_counts(dropna=False)


spending_flag
Low spender     31797024
High spender      635436
Name: count, dtype: int64

In [40]:
# Check the new column
ords_prods_merge.head(20)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge,...,prices,price_range_loc,busiest_day,busiest_day_updated,slowest_day_updated,busiest_period_of_day,max_order,loyalty_flag,avg_price,spending_flag
0,2539329,1,1,2,8,NaN,196,1,0,both,...,9.0,Mid-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
1,2539329,1,1,2,8,NaN,14084,2,0,both,...,12.5,Mid-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
2,2539329,1,1,2,8,NaN,12427,3,0,both,...,4.4,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
3,2539329,1,1,2,8,NaN,26088,4,0,both,...,4.7,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
4,2539329,1,1,2,8,NaN,26405,5,0,both,...,1.0,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
5,2398795,1,2,3,7,15.0,196,1,1,both,...,9.0,Mid-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
6,2398795,1,2,3,7,15.0,10258,2,0,both,...,3.0,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
7,2398795,1,2,3,7,15.0,12427,3,1,both,...,4.4,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
8,2398795,1,2,3,7,15.0,13176,4,0,both,...,10.3,Mid-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender
9,2398795,1,2,3,7,15.0,26088,5,1,both,...,4.7,Low-range product,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender


###  Create an Order Frequency Flag Based on the "days_since_prior_order" Column

In [43]:
# Step 1: Calculate the median 'days_since_prior_order' for each user
ords_prods_merge['median_days'] = ords_prods_merge.groupby('user_id')['days_since_prior_order'].transform('median')

# Step 2: Create the order frequency flag based on the median 'days_since_prior_order'
ords_prods_merge.loc[ords_prods_merge['median_days'] > 20, 'order_frequency_flag'] = 'Non-frequent customer'
ords_prods_merge.loc[(ords_prods_merge['median_days'] <= 20) & (ords_prods_merge['median_days'] > 10), 'order_frequency_flag'] = 'Regular customer'
ords_prods_merge.loc[ords_prods_merge['median_days'] <= 10, 'order_frequency_flag'] = 'Frequent customer'

# Display the result to check the new 'order_frequency_flag' column
ords_prods_merge['order_frequency_flag'].value_counts(dropna=False)


order_frequency_flag
Frequent customer        21576343
Regular customer          7216768
Non-frequent customer     3639349
Name: count, dtype: int64

In [45]:
# Check that the column was created successfully and properly
ords_prods_merge.head(20)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge,...,busiest_day,busiest_day_updated,slowest_day_updated,busiest_period_of_day,max_order,loyalty_flag,avg_price,spending_flag,median_days,order_frequency_flag
0,2539329,1,1,2,8,NaN,196,1,0,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
1,2539329,1,1,2,8,NaN,14084,2,0,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
2,2539329,1,1,2,8,NaN,12427,3,0,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
3,2539329,1,1,2,8,NaN,26088,4,0,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
4,2539329,1,1,2,8,NaN,26405,5,0,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
5,2398795,1,2,3,7,15.0,196,1,1,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
6,2398795,1,2,3,7,15.0,10258,2,0,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
7,2398795,1,2,3,7,15.0,12427,3,1,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
8,2398795,1,2,3,7,15.0,13176,4,0,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer
9,2398795,1,2,3,7,15.0,26088,5,1,both,...,Regularly busy,Busiest days,Regularly busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer


###  Export the DataFrame as a Pickle File

In [48]:
# Set the file path for the pickle file
pickle_file_path = os.path.join(path, 'Data', 'Prepared Data', 'ords_prods_merge_with_flags.pkl')

# Export the dataframe to a pickle file
ords_prods_merge.to_pickle(pickle_file_path)

# Confirm the file was saved successfully
pickle_file_path


'/Users/muhammaddildar/Desktop/03-2025 Instacart Basket Analysis/Data/Prepared Data/ords_prods_merge_with_flags.pkl'